In [1]:
from tools import os_tools

In [2]:
result = os_tools.create_os_command("which process is using the most cpu ??", [])

In [3]:
result

'ps aux --sort=-%cpu | head -n 1'

In [ ]:
# services/agent_runner.py

from agents.criticagent import evaluate_command
from services.confirmation_store import store_pending
from services.executor import execute_command
from services.messaging import send_message  # telegram/whatsapp

async def handle_agent_command(user_id, message, command, context, connector):
    
    critic_result = evaluate_command(
        user_intent=message,
        command=command,
        context=context
    )

    decision = critic_result["decision"]

    if decision == "ALLOW":
        result = execute_command(connector, command)
        await send_message(user_id, f"✅ Done:\n{result}")
        return

    elif decision == "CONFIRM":
        store_pending(user_id, {
            "command": command,
            "message": message
        })

        await send_message(user_id, critic_result["user_message"])
        return

    elif decision == "BLOCK":
        await send_message(user_id, f"❌ Blocked: {critic_result['reason']}")
        return